In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, LSTM, Dropout
from sklearn.metrics import mean_squared_error
from tensorflow.keras.callbacks import EarlyStopping
import keras
import keras_tuner
from tensorflow.keras.optimizers import Adam

In [2]:
# Read datasets
data = pd.read_csv("../data/04_processed/train.csv")

# Split to training and validation set
train = data[data["year"] < 2013].copy()
val = data[data["year"] >= 2012].copy()

In [3]:
# Normalize x,y coords
train["x_norm"] = train["x_coord"] / train["x_coord"].max()
train["y_norm"] = train["y_coord"] / train["x_coord"].max()

val["x_norm"] = val["x_coord"] / train["y_coord"].max() 
val["y_norm"] = val["y_coord"] / train["y_coord"].max() 

In [4]:
train = train.sort_values(by=["x_coord", "y_coord", "date_time"])
val = val.sort_values(by=["x_coord", "y_coord", "date_time"])

In [5]:
# Create lag feature
train["lag_27"] = train.groupby(["x_coord","y_coord"])["water_percentage"].shift(27)
val["lag_27"] = val.groupby(["x_coord","y_coord"])["water_percentage"].shift(27)

train = train.dropna(subset=["lag_27"])
val = val.dropna(subset=["lag_27"])

In [7]:
def create_water_land_data(data, lookback):
    dataX, dataY = [], []
    # group data per grid and per year
    group_data = data.groupby(["x_coord", "y_coord", "year"])

    for _, grid_data_per_year in group_data:
        grid_data_per_year = grid_data_per_year.sort_values("date_time")
        data_values = grid_data_per_year[["water_percentage", "lag_27", "x_norm", "y_norm",
                                "inland_aquaculture", "shrimp_rice_farming", 
                                "single_rice_cropping", "triple_rice_cropping",
                                "double_rice_cropping_dry", "double_rice_cropping_rain",
                                "others"]].values
        for i in range(len(data_values) - lookback):
            a = data_values[i : (i + lookback), :]
            dataX.append(a)
            dataY.append(data_values[i + lookback,0])
        
    return np.array(dataX),np.array(dataY)

In [8]:
# Drop irrelevant columns for training/validation
train = train[["year","water_percentage","lag_27","x_coord","y_coord",
                "inland_aquaculture", "shrimp_rice_farming", 
                "single_rice_cropping", "triple_rice_cropping",
                "double_rice_cropping_dry", "double_rice_cropping_rain",
                "others", "date_time", "x_norm", "y_norm"]]
val = val[["year","water_percentage","lag_27","x_coord","y_coord",
                "inland_aquaculture", "shrimp_rice_farming", 
                "single_rice_cropping", "triple_rice_cropping",
                "double_rice_cropping_dry", "double_rice_cropping_rain",
                "others", "date_time", "x_norm", "y_norm"]]

In [9]:
# Create data with lookback=4
lookback = 4
X_train, y_train = create_water_land_data(train,lookback)
X_val, y_val = create_water_land_data(val,lookback)

In [10]:
def build_model(hp):
    model = Sequential()
    model.add(Input(shape=(4, 11)))
    # 1st LSTM Layer: Changed input_shape to (lookback, 2)
    model.add(LSTM(
        units=hp.Int("units_1", min_value=32, max_value=256, step=32),
        activation="tanh", return_sequences=True))
    model.add(Dropout(hp.Float("dropout_1", 0.1, 0.4, step=0.1)))
    # 2nd LSTM Layer: Processes the sequence
    model.add(LSTM(
        units=hp.Int("units_2", min_value=16, max_value=128, step=16),
        activation="tanh", return_sequences=False))
    model.add(Dropout(hp.Float("dropout_2", 0.1, 0.4, step=0.1)))
    model.add(Dense(1))
    
    # Tuning the Learning Rate
    lr = hp.Choice("learning_rate", values=[1e-2, 1e-3, 1e-4])
    model.compile(optimizer=Adam(learning_rate=lr), loss="mean_squared_error")
    return model

In [11]:
# Initialize Bayesian Optimizer
tuner = keras_tuner.BayesianOptimization(
    build_model,
    objective="val_loss",
    max_trials=30,
    seed=123,
    executions_per_trial=1,
    directory="tuning_results",
    project_name="water_level_prediction_with_land")

In [12]:
early_stop = EarlyStopping(monitor='val_loss', patience=5)
tuner.search(
    X_train, y_train,
    epochs=50,
    validation_data=(X_val, y_val), # This enforces the temporal split
    verbose=1,
    callbacks=[early_stop]
)

Trial 30 Complete [00h 01m 15s]
val_loss: 0.005448433104902506

Best val_loss So Far: 0.005150956101715565
Total elapsed time: 00h 26m 19s


In [13]:
# 1. Get the best hyperparameters object
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

# 2. Rebuild the model using the OBJECT, not the keys
best_model = tuner.hypermodel.build(best_hps)

# 3. Use .values.get() to safely print parameters
print(f"""
Optimal Configuration found:
- Units 1:      {best_hps.values.get("units_1", 'N/A')}
- Units 2:      {best_hps.values.get("units_2", 'N/A')}
- Learning Rate: {best_hps.values.get("learning_rate", 'N/A')}
- Dropout 1:     {best_hps.values.get("dropout_1", 'N/A')}
- Dropout 2:     {best_hps.values.get("dropout_2", 'N/A')}
""")


Optimal Configuration found:
- Units 1:      160
- Units 2:      80
- Learning Rate: 0.001
- Dropout 1:     0.4
- Dropout 2:     0.4

